# PaperLens: pipeline notebook

Fine-tuning **Qwen2.5-3B-Instruct** with QLoRA to turn an ML paper's title and abstract into structured JSON (task, method, datasets, metrics, main result, contribution type). Results and write-up are in the repository README.

**Before you run anything**
- Platform: Kaggle notebook. Cells that load the model (baselines, training, evaluation, latency) need a free **T4 GPU**; the others do not. Turn on **Internet**.
- Kaggle Secrets (Add-ons, Secrets): `HF_TOKEN` (write access), `GEMINI_API_KEY` (`GEMINI_API_KEY_2` and `_3` optional), `GROQ_API_KEY` (only for the gpt-oss-20b comparison).
- Most cells restore what they need from the private Hugging Face repo `<username>/paperlens-data`, so sections can run in separate sessions. Some cells hard-code the author's username `punit-09`: replace it with yours.
- **Do not re-run the sampling cell.** The arXiv snapshot changes daily, so a re-run draws a different sample. The exact data used is in the public dataset repo.
- Cell outputs were cleared before publishing.

Run once at the start of a fresh session, then **restart the kernel** before running the cells below (importing torch before the install can break the session).

In [ ]:
!pip -q install datasets
!pip -q install -U huggingface_hub google-genai pydantic accelerate "peft==0.21.2" "bitsandbytes==0.50.2"


## 1. Data: sample arXiv abstracts
Random sample of 2023 to 2026 papers in cs.LG, cs.CL and cs.CV with 100 to 300 word abstracts (seed 42): 1,500 train, 100 validation, 150 test. Then back the raw files up to Hugging Face.

In [ ]:
import json, random, re, os
from collections import Counter
from datasets import load_dataset

TARGET = {"cs.LG", "cs.CL", "cs.CV"}
ds = load_dataset("librarian-bots/arxiv-metadata-snapshot",
                  split="train", streaming=True)

pool = []
for r in ds:
    if not (set(r["categories"].split()) & TARGET):
        continue
    if not re.match(r"^(23|24|25|26)\d{2}\.\d{4,5}$", r["id"]):
        continue
    abstract = " ".join(r["abstract"].split())
    if not (100 <= len(abstract.split()) <= 300):
        continue
    pool.append({"id": r["id"], "title": r["title"],
                 "categories": r["categories"], "abstract": abstract})

print("candidates:", len(pool))

random.seed(42)
random.shuffle(pool)
splits = {"test": pool[:150], "val": pool[150:250], "train": pool[250:1750]}

os.makedirs("/kaggle/working/data", exist_ok=True)
for name, rows in splits.items():
    with open(f"/kaggle/working/data/{name}_raw.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print(name, len(rows))

print(Counter(c for r in splits["train"] for c in r["categories"].split() if c in TARGET))
print(splits["train"][0]["abstract"][:300])


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, login, snapshot_download

login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi()
REPO = f"{api.whoami()['name']}/paperlens-data"
api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)

def backup(local="/kaggle/working/data"):
    api.upload_folder(folder_path=local, repo_id=REPO,
                      repo_type="dataset", path_in_repo="data")
    print("backed up to", REPO)

def restore():
    snapshot_download(REPO, repo_type="dataset", local_dir="/kaggle/working")
    print("restored")

backup()


## 2. Labels
Gemini labels each title and abstract with one fixed rule prompt; code-side checks then enforce faithfulness (every dataset and metric must appear in the text, no model names as datasets, numbers must exist in the abstract). Resumable.

In [ ]:
import json, os, re, time, random, threading, itertools
from typing import Literal
from concurrent.futures import ThreadPoolExecutor, as_completed
from pydantic import BaseModel
from google import genai
from google.genai import types
from huggingface_hub import HfApi, login, snapshot_download
from kaggle_secrets import UserSecretsClient

# ---------- config ----------
DRY_RUN = False    # False = full run on test, val and train
MODEL, FALLBACK = "gemini-3.5-flash-lite", "gemini-3.1-flash-lite"
KEY_NAMES = ["GEMINI_API_KEY", "GEMINI_API_KEY_2", "GEMINI_API_KEY_3"]
DATA = "/kaggle/working/data"
SUFFIX = "_labeled_v3"

# ---------- clients + backup ----------
sec = UserSecretsClient()
clients = []
for n in KEY_NAMES:
    try: clients.append(genai.Client(api_key=sec.get_secret(n)))
    except Exception: print(f"secret {n} not found, skipping")
assert clients, "No Gemini keys found in Kaggle Secrets"
WORKERS, RPM = 4 * len(clients), 30 * len(clients)
print(f"{len(clients)} key(s), {WORKERS} workers, {RPM} req/min cap, DRY_RUN={DRY_RUN}")
_rr = itertools.count()

login(sec.get_secret("HF_TOKEN"))
api = HfApi(); REPO = f"{api.whoami()['name']}/paperlens-data"
_up = threading.Lock()
def _upload(block):
    if not _up.acquire(blocking=block): return
    try:
        api.upload_folder(folder_path=DATA, repo_id=REPO, repo_type="dataset", path_in_repo="data")
        print("  [backed up]")
    except Exception as e:
        print("  backup failed:", str(e)[:80])
    finally:
        _up.release()
def backup(sync=False):
    if sync: _upload(True)
    else: threading.Thread(target=_upload, args=(False,), daemon=True).start()

if not os.path.exists(f"{DATA}/test_raw.jsonl"):
    snapshot_download(REPO, repo_type="dataset", local_dir="/kaggle/working")
    print("restored data from HF")
if not DRY_RUN and os.path.exists(f"{DATA}/dry_v3.jsonl"):
    os.remove(f"{DATA}/dry_v3.jsonl")

# ---------- schema + v3 prompt ----------
class Paper(BaseModel):
    task: str
    method: str
    datasets: list[str]
    metrics: list[str]
    main_result: str
    contribution_type: Literal["new_method","benchmark_or_dataset","analysis","survey","other"]

SYSTEM = """You extract structured info from a machine learning paper (title + abstract).
Use ONLY what the text says. Never guess or add outside knowledge.
- task: lowercase noun phrase, max 10 words: the problem the paper addresses.
- method: the authors' OWN proposed approach, max 12 words. Use its name if given; if both a full name and an acronym are given, write "Full Name (ACRONYM)". Never list baselines, backbones, tools or pretrained models that the authors merely use. If the paper is an analysis or survey with no new method, describe what is studied (e.g. "empirical study of color transformations for registration").
- datasets: only names of datasets or benchmarks that appear in the text. Never models or architectures. [] if none are named.
- metrics: only named evaluation metrics that appear in the text (e.g. F1, BLEU, WER, mAP, accuracy, FLOPs). Do not infer a metric: if the text only says "performance" or gives percentages without naming a metric, use []. No vague concepts (cost, efficiency, performance, robustness). List each metric once (acronym OR full name). Max 5.
- main_result: ONE sentence stating the paper's main empirical finding, copying numbers exactly from the text. If there are no numbers, state the main qualitative finding. Do not describe the method here.
- contribution_type: new_method (proposes a new method/model/technique) | benchmark_or_dataset (main contribution is a new dataset or benchmark) | analysis (empirical or theoretical study of existing methods) | survey (literature review) | other."""

# ---------- code-side sanitizing ----------
MODELS = re.compile(r"\b(resnet|bert|roberta|llama|gpt|deit|vit|t5|qwen|mistral|clip|yolo|gemma)\b", re.I)
GENERIC = {"performance","cost","computational cost","efficiency","robustness","quality",
           "speed","memory","fps","complexity","scalability","effectiveness"}
compact = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())

def grounded(item, text):
    c = compact(item)
    if not c: return False
    if len(c) >= 4: return c in compact(text)
    return re.search(r"(?<![A-Za-z0-9])" + re.escape(item.strip()) + r"(?![A-Za-z0-9])", text) is not None

def dedupe(xs):
    seen, out = set(), []
    for x in xs:
        k = x.strip().lower()
        if k and k not in seen: seen.add(k); out.append(x.strip())
    return out

def sanitize(l, title, abstract):
    text = title + " " + abstract
    task = l["task"].strip().rstrip(".")
    if len(task) > 1 and not task[1].isupper(): task = task[0].lower() + task[1:]
    method = l["method"].strip()
    result = " ".join(l["main_result"].split())
    if not task or not method or not result: raise ValueError("empty field")
    if len(task.split()) > 15 or len(method.split()) > 15: raise ValueError("task/method too long")
    if len(result.split()) > 60: raise ValueError("main_result too long")
    nums = re.findall(r"\d+(?:\.\d+)?", result)
    if any(n not in text for n in nums): raise ValueError("number in main_result not in abstract")
    datasets = dedupe([d for d in l["datasets"] if grounded(d, text) and not MODELS.search(d)])
    metrics = dedupe([m for m in l["metrics"] if grounded(m, text) and m.strip().lower() not in GENERIC])[:5]
    return dict(task=task, method=method, datasets=datasets, metrics=metrics,
                main_result=result, contribution_type=l["contribution_type"])

# ---------- API call with rate limiter + retries ----------
def label_once(model, title, abstract, temp, thinking=True):
    cfg = dict(system_instruction=SYSTEM, temperature=temp,
               response_mime_type="application/json", response_schema=Paper,
               http_options=types.HttpOptions(timeout=40000))
    if thinking: cfg["thinking_config"] = types.ThinkingConfig(thinking_level="minimal")
    c = clients[next(_rr) % len(clients)]
    r = c.models.generate_content(model=model, contents=f"Title: {title}\nAbstract: {abstract}",
                                  config=types.GenerateContentConfig(**cfg))
    return json.loads(r.text)

_lock, _next = threading.Lock(), [0.0]
def throttle():
    with _lock:
        now = time.time(); wait = max(0, _next[0] - now)
        _next[0] = max(now, _next[0]) + 60 / RPM
    if wait: time.sleep(wait)

def call(row):
    last, kind = None, "api"
    for attempt in range(6):
        model = MODEL if attempt in (0, 1, 4) else FALLBACK
        temp = 0 if attempt == 0 else 0.4
        throttle()
        try:
            try: raw = label_once(model, row["title"], row["abstract"], temp)
            except Exception as e:
                if any(k in str(e).lower() for k in ["400", "thinking", "validation"]):
                    raw = label_once(model, row["title"], row["abstract"], temp, thinking=False)
                else: raise
            clean = sanitize(Paper(**raw).model_dump(), row["title"], row["abstract"])
            return {**clean, "_model": model}
        except ValueError as e:
            last, kind = str(e)[:100], "invalid"; time.sleep(0.5)
        except Exception as e:
            last, kind = str(e)[:120], "api"; time.sleep(min(60, 2 ** attempt) + random.random())
    return {"_error": last, "_kind": kind}

# ---------- main loop (resumable) ----------
def run_split(name):
    raw = [json.loads(l) for l in open(f"{DATA}/{name}_raw.jsonl")]
    out_path = f"{DATA}/{name}{SUFFIX}.jsonl"
    if DRY_RUN:
        raw = raw[:12]; out_path = f"{DATA}/dry_v3.jsonl"
        if os.path.exists(out_path): os.remove(out_path)
    done = set()
    if os.path.exists(out_path):
        for l in open(out_path):
            try: done.add(json.loads(l)["id"])
            except Exception: pass
    todo = [r for r in raw if r["id"] not in done]
    print(f"{name}: {len(done)} already done, {len(todo)} to do")
    if not todo: return True
    n = api_errs = dropped = 0
    ex = ThreadPoolExecutor(WORKERS); f = open(out_path, "a")
    futs = {ex.submit(call, r): r for r in todo}
    ok = True
    for fu in as_completed(futs):
        res = fu.result()
        if "_error" in res:
            if res["_kind"] == "invalid": dropped += 1
            else:
                api_errs += 1
                if api_errs == 1 or api_errs % 5 == 0: print("  api error:", res["_error"])
                if api_errs >= 15:
                    print("Too many API errors (quota or outage?). Stopping; progress is saved.")
                    ex.shutdown(wait=False, cancel_futures=True); ok = False; break
            continue
        f.write(json.dumps({**futs[fu], "label": res}) + "\n"); f.flush(); n += 1
        if n % 100 == 0: print(f"  {name}: {n} labeled, {dropped} dropped, {api_errs} api errors"); backup()
    f.close(); ex.shutdown(wait=False)
    print(f"{name}: +{n} labeled, {dropped} dropped (failed checks), {api_errs} api errors")
    backup(sync=True)
    return ok

finished = True
for split in (["test"] if DRY_RUN else ["test", "val", "train"]):
    if not run_split(split): finished = False; break

if DRY_RUN and finished:
    for l in open(f"{DATA}/dry_v3.jsonl"):
        r = json.loads(l); lab = r["label"]
        print("=" * 70); print(r["title"])
        for k in ["task", "method", "datasets", "metrics", "main_result", "contribution_type"]:
            print(f"  {k}: {lab[k]}")
print("DONE" if finished else "STOPPED EARLY: rerun this cell to resume")


## 3. Final dataset files
Manual corrections on two reviewed test rows, removal of count-style dataset entries, and export of the chat-format train, validation and test files.

In [ ]:
import json, re, os
from huggingface_hub import HfApi, login, snapshot_download
from kaggle_secrets import UserSecretsClient

DATA = "/kaggle/working/data"
login(UserSecretsClient().get_secret("HF_TOKEN"))
REPO = f"{HfApi().whoami()['name']}/paperlens-data"
if not os.path.exists(f"{DATA}/test_labeled_v3.jsonl"):
    snapshot_download(REPO, repo_type="dataset", local_dir="/kaggle/working")

def load(s): return [json.loads(l) for l in open(f"{DATA}/{s}_labeled_v3.jsonl")]
sets = {s: load(s) for s in ["train", "val", "test"]}

# 1) manual corrections on the reviewed test rows
test = sets["test"]
for m in ["MMrTRE", "AMrTRE"]:
    if m not in test[2]["label"]["metrics"]: test[2]["label"]["metrics"].append(m)
test[25]["label"]["method"] = "NMIXX"
print("#2 ", test[2]["title"][:60], "->", test[2]["label"]["metrics"])
print("#25", test[25]["title"][:60], "->", test[25]["label"]["method"])

# 2) drop count-style "datasets" like "20 tissue sample pairs"
COUNT = re.compile(r"^\d[\d,\.]*\s+(\w+\s+)?(pairs|samples|images|patients|subjects|participants|cases|scans|videos|examples)$", re.I)
for rows in sets.values():
    for r in rows:
        r["label"]["datasets"] = [d for d in r["label"]["datasets"] if not COUNT.match(d.strip())]

# 3) no overlap between splits
ids = {s: {r["id"] for r in rows} for s, rows in sets.items()}
assert not (ids["train"] & ids["test"]) and not (ids["train"] & ids["val"]) and not (ids["val"] & ids["test"]), "split overlap!"

# 4) export chat-format files
INSTR = ("Extract structured information from this ML paper. Reply with JSON only, "
         "with keys: task, method, datasets, metrics, main_result, contribution_type.")
KEYS = ["task", "method", "datasets", "metrics", "main_result", "contribution_type"]
os.makedirs(f"{DATA}/final", exist_ok=True)
for s, rows in sets.items():
    with open(f"{DATA}/final/{s}.jsonl", "w") as f:
        for i, r in enumerate(rows):
            d = {"id": r["id"], "messages": [
                {"role": "system", "content": INSTR},
                {"role": "user", "content": f"Title: {r['title']}\nAbstract: {r['abstract']}"},
                {"role": "assistant", "content": json.dumps({k: r["label"][k] for k in KEYS}, ensure_ascii=False)}]}
            if s == "test": d["reviewed"] = i < 30
            f.write(json.dumps(d, ensure_ascii=False) + "\n")
    print(s, len(rows), "rows exported")

HfApi().upload_folder(folder_path=DATA, repo_id=REPO, repo_type="dataset", path_in_repo="data")
print("backed up")
print(open(f"{DATA}/final/train.jsonl").readline()[:700])


## 4. Scoring code
Valid-JSON check, token F1 and set F1 per field, tested on a perfect, a lazy and a broken predictor.

In [ ]:
%%writefile /kaggle/working/paperlens_eval.py
import json, re
from collections import Counter

KEYS = ["task", "method", "datasets", "metrics", "main_result", "contribution_type"]
TYPES = {"new_method", "benchmark_or_dataset", "analysis", "survey", "other"}

def parse(text):
    """Dict if text is valid JSON with the right schema, else None."""
    try:
        t = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
        d = json.loads(t)
    except Exception:
        return None
    if not isinstance(d, dict) or set(d) != set(KEYS): return None
    if not all(isinstance(d[k], str) for k in ["task", "method", "main_result", "contribution_type"]): return None
    if not all(isinstance(d[k], list) and all(isinstance(x, str) for x in d[k]) for k in ["datasets", "metrics"]): return None
    if d["contribution_type"] not in TYPES: return None
    return d

toks = lambda s: re.findall(r"[a-z0-9]+", s.lower())
compact = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())
nums = lambda s: re.findall(r"\d+(?:\.\d+)?", s)

def f1_from_counters(p, g):
    if not p and not g: return 1.0
    overlap = sum((p & g).values())
    if overlap == 0: return 0.0
    pr, rc = overlap / sum(p.values()), overlap / sum(g.values())
    return 2 * pr * rc / (pr + rc)

def tok_f1(p, g): return f1_from_counters(Counter(toks(p)), Counter(toks(g)))
def set_f1(p, g): return f1_from_counters(Counter({compact(x) for x in p if compact(x)}),
                                          Counter({compact(x) for x in g if compact(x)}))
def num_f1(p, g): return f1_from_counters(Counter(nums(p)), Counter(nums(g)))

def evaluate(preds, gold_rows, subset="all"):
    """preds: {id: raw model text}. gold_rows: rows of final/test.jsonl. subset: 'all' or 'reviewed'."""
    rows = [r for r in gold_rows if subset == "all" or r.get("reviewed")]
    names = ["valid", "task_f1", "method_f1", "datasets_f1", "metrics_f1",
             "datasets_f1_nonempty", "metrics_f1_nonempty", "type_acc",
             "result_num_f1", "result_tok_f1"]
    acc = {k: [] for k in names}
    for r in rows:
        g = json.loads(r["messages"][-1]["content"])
        p = parse(preds.get(r["id"], ""))
        sc = lambda fn, *a: 0.0 if p is None else fn(*a)
        acc["valid"].append(p is not None)
        acc["task_f1"].append(sc(tok_f1, p["task"] if p else "", g["task"]))
        acc["method_f1"].append(sc(tok_f1, p["method"] if p else "", g["method"]))
        acc["type_acc"].append(float(p is not None and p["contribution_type"] == g["contribution_type"]))
        for k in ["datasets", "metrics"]:
            s = sc(set_f1, p[k] if p else [], g[k])
            acc[f"{k}_f1"].append(s)
            if g[k]: acc[f"{k}_f1_nonempty"].append(s)
        acc["result_tok_f1"].append(sc(tok_f1, p["main_result"] if p else "", g["main_result"]))
        if nums(g["main_result"]):
            acc["result_num_f1"].append(sc(num_f1, p["main_result"] if p else "", g["main_result"]))
    out = {k: round(sum(v) / max(1, len(v)), 3) for k, v in acc.items()}
    out["n"] = len(rows)
    return out


In [ ]:
import json, os, sys, shutil
sys.path.insert(0, "/kaggle/working")
from huggingface_hub import HfApi, login, snapshot_download
from kaggle_secrets import UserSecretsClient
import paperlens_eval as pe

DATA = "/kaggle/working/data"
login(UserSecretsClient().get_secret("HF_TOKEN"))
REPO = f"{HfApi().whoami()['name']}/paperlens-data"
if not os.path.exists(f"{DATA}/final/test.jsonl"):
    snapshot_download(REPO, repo_type="dataset", local_dir="/kaggle/working")

gold = [json.loads(l) for l in open(f"{DATA}/final/test.jsonl")]

perfect = {r["id"]: r["messages"][-1]["content"] for r in gold}
lazy = {r["id"]: json.dumps({"task": "", "method": "", "datasets": [], "metrics": [],
                             "main_result": "", "contribution_type": "new_method"}) for r in gold}
broken = {r["id"]: "hello" for r in gold}

for name, preds in [("perfect (should be all 1.0)", perfect),
                    ("lazy: always empty + new_method", lazy),
                    ("broken: not JSON", broken)]:
    print(name); print("  all :", pe.evaluate(preds, gold, "all"))
print("reviewed slice, perfect:", pe.evaluate(perfect, gold, "reviewed")["n"], "rows")

shutil.copy("/kaggle/working/paperlens_eval.py", f"{DATA}/paperlens_eval.py")
HfApi().upload_folder(folder_path=DATA, repo_id=REPO, repo_type="dataset", path_in_repo="data")
print("backed up")


## 5. Baselines: untuned Qwen2.5-3B-Instruct
First with a short instruction, then with the full schema in the prompt.

In [ ]:
import json, os, sys, time, torch
from huggingface_hub import HfApi, login, snapshot_download
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "GPU is off: Settings -> Accelerator -> GPU T4 x2"

DATA = "/kaggle/working/data"
login(UserSecretsClient().get_secret("HF_TOKEN"))
REPO = f"{HfApi().whoami()['name']}/paperlens-data"
if not os.path.exists(f"{DATA}/final/test.jsonl"):
    snapshot_download(REPO, repo_type="dataset", local_dir="/kaggle/working")
sys.path.insert(0, DATA)
import paperlens_eval as pe

MODEL = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map={"": 0})
model.eval()

gold = [json.loads(l) for l in open(f"{DATA}/final/test.jsonl")]
out_path = f"{DATA}/preds_base.jsonl"
done = {}
if os.path.exists(out_path):
    for l in open(out_path):
        d = json.loads(l); done[d["id"]] = d["pred"]
todo = [r for r in gold if r["id"] not in done]
print(len(done), "already done,", len(todo), "to do")

BS, t0 = 8, time.time()
f = open(out_path, "a")
for i in range(0, len(todo), BS):
    batch = todo[i:i + BS]
    prompts = [tok.apply_chat_template(r["messages"][:2], tokenize=False,
                                       add_generation_prompt=True) for r in batch]
    enc = tok(prompts, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=400, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    for r, t in zip(batch, texts):
        f.write(json.dumps({"id": r["id"], "pred": t}) + "\n"); done[r["id"]] = t
    f.flush()
    print(f"{len(done)}/{len(gold)}  {time.time() - t0:.0f}s")
f.close()

print("\nALL :", pe.evaluate(done, gold, "all"))
print("REVIEWED:", pe.evaluate(done, gold, "reviewed"))
HfApi().upload_folder(folder_path=DATA, repo_id=REPO, repo_type="dataset", path_in_repo="data")
print("backed up")
print("\nSAMPLE OUTPUT:\n", done[gold[0]["id"]][:600])


In [ ]:
import json, os, re, sys, time, torch
from huggingface_hub import HfApi, login, snapshot_download
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer, AutoModelForCausalLM

DATA = "/kaggle/working/data"
login(UserSecretsClient().get_secret("HF_TOKEN"))
REPO = f"{HfApi().whoami()['name']}/paperlens-data"
if not os.path.exists(f"{DATA}/final/test.jsonl"):
    snapshot_download(REPO, repo_type="dataset", local_dir="/kaggle/working")
sys.path.insert(0, DATA)
import paperlens_eval as pe

if "model" not in globals():          # session was restarted: reload the model
    MODEL = "Qwen/Qwen2.5-3B-Instruct"
    tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map={"": 0})
    model.eval()

gold = [json.loads(l) for l in open(f"{DATA}/final/test.jsonl")]

# ---- why did baseline 1 fail? ----
def load_preds(p): return {json.loads(l)["id"]: json.loads(l)["pred"] for l in open(p)}
base1 = load_preds(f"{DATA}/preds_base.jsonl")
c = dict(json_parses=0, has_exact_keys=0, fully_valid=0)
for t in base1.values():
    t2 = re.sub(r"^```(?:json)?|```$", "", t.strip(), flags=re.M).strip()
    try: d = json.loads(t2)
    except Exception: continue
    c["json_parses"] += 1
    if isinstance(d, dict) and set(d) == set(pe.KEYS): c["has_exact_keys"] += 1
    if pe.parse(t) is not None: c["fully_valid"] += 1
print("baseline 1 breakdown (out of 150):", c)

# ---- baseline 2: full schema prompt ----
SYS2 = """You extract structured information from an ML paper (title + abstract).
Reply with ONE JSON object and nothing else (no markdown fences, no comments). Keys:
- task: string, lowercase noun phrase, max 10 words
- method: string, the authors' own proposed method (use its name if given), max 12 words
- datasets: list of strings, names of datasets or benchmarks mentioned; [] if none
- metrics: list of strings, named evaluation metrics mentioned; [] if none
- main_result: ONE sentence (a string) with the main empirical finding, keeping numbers
- contribution_type: exactly one of "new_method", "benchmark_or_dataset", "analysis", "survey", "other"
Use only information in the text."""

out_path = f"{DATA}/preds_base_prompted.jsonl"
done = load_preds(out_path) if os.path.exists(out_path) else {}
todo = [r for r in gold if r["id"] not in done]
print(len(done), "already done,", len(todo), "to do")

BS, t0 = 8, time.time()
f = open(out_path, "a")
for i in range(0, len(todo), BS):
    batch = todo[i:i + BS]
    prompts = [tok.apply_chat_template(
        [{"role": "system", "content": SYS2}, r["messages"][1]],
        tokenize=False, add_generation_prompt=True) for r in batch]
    enc = tok(prompts, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=400, do_sample=False, pad_token_id=tok.eos_token_id)
    texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    for r, t in zip(batch, texts):
        f.write(json.dumps({"id": r["id"], "pred": t}) + "\n"); done[r["id"]] = t
    f.flush(); print(f"{len(done)}/{len(gold)}  {time.time() - t0:.0f}s")
f.close()

print("\nBASELINE 2 (prompted) ALL :", pe.evaluate(done, gold, "all"))
print("BASELINE 2 (prompted) REVIEWED:", pe.evaluate(done, gold, "reviewed"))
HfApi().upload_folder(folder_path=DATA, repo_id=REPO, repo_type="dataset", path_in_repo="data")
print("backed up")
print("\nSAMPLE:\n", done[gold[0]["id"]][:500])


## 6. QLoRA fine-tuning
4-bit NF4 base, LoRA r=16 on all attention and MLP projections, 2 epochs (186 steps), loss on the JSON answer only. Checkpoints go to Hugging Face and training resumes after a crash. Set `SMOKE = True` for a 3-step test first.

In [ ]:
import os, gc, sys, json, math, time, random, shutil
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np, torch, torch.nn.functional as F
from huggingface_hub import HfApi, login, snapshot_download
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel

# ============ CONFIG ============
SMOKE = False           # True = 3-step test, nothing saved. Set False for the real run.
MODEL = "Qwen/Qwen2.5-3B-Instruct"
EPOCHS, LR = 2, 2e-4
MB, GA = 4, 4           # micro-batch 4 x accumulation 4 = effective batch 16. If OOM: MB=2, GA=8
R, ALPHA, DROPOUT = 16, 32, 0.05
SEED, MAXLEN, SAVE_EVERY = 42, 1024, 25
CFG = dict(MODEL=MODEL, EPOCHS=EPOCHS, LR=LR, R=R, SEED=SEED)
DATA = "/kaggle/working/data"
CKPT_SAVE, CKPT_DL, FINAL = "/kaggle/working/ckpt_save", "/kaggle/working/ckpt_dl", "/kaggle/working/final_adapter"

# ============ free GPU memory from earlier cells ============
for n in ["model", "base", "base_model", "opt", "scaler", "sched", "enc", "out"]:
    globals().pop(n, None)
gc.collect(); torch.cuda.empty_cache()

assert torch.cuda.is_available(), "GPU is off: Settings -> Accelerator -> GPU T4 x2"
cap = torch.cuda.get_device_capability(0)
use_bf16 = cap[0] >= 8
DT = torch.bfloat16 if use_bf16 else torch.float16
print(torch.cuda.get_device_name(0), cap, "bf16" if use_bf16 else "fp16", "| SMOKE =", SMOKE)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ============ HF login + data ============
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi(); USER = api.whoami()["name"]
DATA_REPO, CKPT_REPO = f"{USER}/paperlens-data", f"{USER}/paperlens-ckpt"
if not os.path.exists(f"{DATA}/final/train.jsonl"):
    snapshot_download(DATA_REPO, repo_type="dataset", local_dir="/kaggle/working")

tok = AutoTokenizer.from_pretrained(MODEL)
assert tok.pad_token_id is not None

def encode(row):
    prompt = tok.apply_chat_template(row["messages"][:2], tokenize=False, add_generation_prompt=True)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    a = tok(row["messages"][2]["content"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
    return p + a, [-100] * len(p) + a          # loss only on the answer

def load_split(s):
    out = []
    for l in open(f"{DATA}/final/{s}.jsonl"):
        ids, lab = encode(json.loads(l))
        if len(ids) <= MAXLEN: out.append((ids, lab))
    return out

train, val = load_split("train"), load_split("val")
if SMOKE: val = val[:16]

def make_batches(epoch):
    """Deterministic per epoch: shuffle, sort inside chunks of 128 by length, batch, shuffle batches."""
    rng = random.Random(SEED + epoch)
    idx = list(range(len(train))); rng.shuffle(idx)
    mbs = []
    for i in range(0, len(idx), 128):
        chunk = sorted(idx[i:i + 128], key=lambda j: len(train[j][0]))
        mbs += [chunk[k:k + MB] for k in range(0, len(chunk), MB)]
    rng.shuffle(mbs)
    return mbs

def collate(data, ids_):
    L = max(len(data[j][0]) for j in ids_)
    x = torch.full((len(ids_), L), tok.pad_token_id, dtype=torch.long)
    y = torch.full((len(ids_), L), -100, dtype=torch.long)
    m = torch.zeros((len(ids_), L), dtype=torch.long)
    for r, j in enumerate(ids_):
        a, b = data[j]; n = len(a)
        x[r, :n] = torch.tensor(a); y[r, :n] = torch.tensor(b); m[r, :n] = 1
    return x.cuda(), m.cuda(), y.cuda()

SPE = len(make_batches(0)) // GA
TOTAL = 3 if SMOKE else EPOCHS * SPE
print(f"{len(train)} train, {len(val)} val | {SPE} steps/epoch | {TOTAL} total steps")

# ============ resume discovery ============
resume, step0 = None, 0
if not SMOKE:
    api.create_repo(CKPT_REPO, repo_type="model", private=True, exist_ok=True)
    if not os.path.exists(f"{CKPT_SAVE}/meta.json"):
        try: snapshot_download(CKPT_REPO, repo_type="model", local_dir=CKPT_DL, allow_patterns=["latest/*"])
        except Exception as e: print("no checkpoint downloaded (fine on a first run):", str(e)[:80])
    cands = [d for d in [CKPT_SAVE, f"{CKPT_DL}/latest"] if os.path.exists(f"{d}/meta.json")]
    if cands:
        resume = max(cands, key=lambda d: json.load(open(f"{d}/meta.json"))["step"])
        meta = json.load(open(f"{resume}/meta.json")); step0 = meta["step"]
        bad = {k: (meta["cfg"].get(k), CFG[k]) for k in CFG if meta["cfg"].get(k) != CFG[k]}
        assert not bad, f"config changed since the checkpoint was saved: {bad}. Restore the old values."
        print(f"RESUMING from step {step0} ({resume})")

# ============ model ============
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DT)
try:
    base_model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, dtype=DT, device_map={"": 0})
except TypeError:
    base_model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, torch_dtype=DT, device_map={"": 0})
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
base_model.enable_input_require_grads()

if resume:
    model = PeftModel.from_pretrained(base_model, resume, is_trainable=True)
else:
    model = get_peft_model(base_model, LoraConfig(
        r=R, lora_alpha=ALPHA, lora_dropout=DROPOUT, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()
inner = model.get_base_model(); core, lm_head = inner.model, inner.lm_head

params = [p for p in model.parameters() if p.requires_grad]
try: opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0, fused=True)
except Exception: opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
WARM = max(1, int(0.06 * TOTAL))
def lr_lambda(s):
    if s < WARM: return (s + 1) / WARM
    p = (s - WARM) / max(1, TOTAL - WARM)
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * p))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=not use_bf16)
if resume:
    st = torch.load(f"{resume}/training_state.pt", map_location="cpu", weights_only=False)
    opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])

# ============ loss / val / checkpoint ============
def seq_loss(x, m, y):
    with torch.autocast("cuda", dtype=DT):
        h = core(input_ids=x, attention_mask=m, use_cache=False).last_hidden_state[:, :-1]
        tgt = y[:, 1:]; sel = tgt != -100
        logits = lm_head(h[sel])                   # vocabulary projection ONLY on answer tokens
    return F.cross_entropy(logits.float(), tgt[sel]), int(sel.sum())

@torch.no_grad()
def val_loss():
    model.eval(); tot = n = 0
    order = sorted(range(len(val)), key=lambda j: len(val[j][0]))
    for i in range(0, len(order), 8):
        l, c = seq_loss(*collate(val, order[i:i + 8])); tot += l.item() * c; n += c
    model.train(); return tot / max(1, n)

def save_ckpt(step):
    tmp = CKPT_SAVE + "_tmp"
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    model.save_pretrained(tmp)
    torch.save({"opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict()},
               f"{tmp}/training_state.pt")
    json.dump({"step": step, "cfg": CFG}, open(f"{tmp}/meta.json", "w"))   # written last
    shutil.rmtree(CKPT_SAVE, ignore_errors=True); os.rename(tmp, CKPT_SAVE)
    try:
        api.upload_folder(folder_path=CKPT_SAVE, repo_id=CKPT_REPO, repo_type="model", path_in_repo="latest")
        print(f"  [checkpoint step {step} saved + uploaded]")
    except Exception as e:
        print(f"  [checkpoint step {step} saved locally; upload failed: {str(e)[:80]}]")

# ============ training loop ============
model.train()
step, t0, s0, nan_streak = step0, time.time(), step0, 0
for ep in range(step0 // SPE, EPOCHS):
    mbs = make_batches(ep)
    while step < min((ep + 1) * SPE, TOTAL):
        k = (step - ep * SPE) * GA
        tot, ok = 0.0, 0
        for ids_ in mbs[k:k + GA]:
            loss, _ = seq_loss(*collate(train, ids_))
            if not torch.isfinite(loss):
                nan_streak += 1
                if nan_streak >= 10: raise RuntimeError("loss is NaN/inf repeatedly: stop and tell me")
                continue
            nan_streak = 0
            scaler.scale(loss / GA).backward()
            tot += loss.item(); ok += 1
        if ok == 0:
            opt.zero_grad(set_to_none=True); sched.step(); step += 1; continue
        scaler.unscale_(opt)
        gn = float(torch.nn.utils.clip_grad_norm_(params, 1.0))
        scaler.step(opt); scaler.update(); sched.step()
        opt.zero_grad(set_to_none=True); step += 1
        done, el = step - s0, time.time() - t0
        if step % 5 == 0 or step == TOTAL or done <= 3:
            print(f"step {step}/{TOTAL} ep {ep+1} loss {tot/ok:.4f} lr {sched.get_last_lr()[0]:.2e} gn {gn:.2f} "
                  f"| {el/done:.1f}s/step ETA {(TOTAL-step)*el/done/60:.0f}min "
                  f"| peak mem {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
        if not SMOKE and step % SAVE_EVERY == 0 and step < TOTAL and step != (ep + 1) * SPE:
            save_ckpt(step)
    if not SMOKE and step == (ep + 1) * SPE:
        print(f"== epoch {ep+1} done | val loss {val_loss():.4f}")
        save_ckpt(step)

vl = val_loss(); print(f"FINAL val loss: {vl:.4f}")

# ============ save final adapter + quick generation test ============
if not SMOKE:
    shutil.rmtree(FINAL, ignore_errors=True); model.save_pretrained(FINAL)
    api.upload_folder(folder_path=FINAL, repo_id=CKPT_REPO, repo_type="model", path_in_repo="final")
    print("final adapter uploaded to", CKPT_REPO, "-> final/")
try:
    model.eval()
    row = json.loads(open(f"{DATA}/final/test.jsonl").readline())
    prompt = tok.apply_chat_template(row["messages"][:2], tokenize=False, add_generation_prompt=True)
    e = tok(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad(), torch.autocast("cuda", dtype=DT):
        o = model.generate(**e, max_new_tokens=300, do_sample=False, pad_token_id=tok.pad_token_id, use_cache=True)
    print("\nSAMPLE OUTPUT:\n", tok.decode(o[0, e["input_ids"].shape[1]:], skip_special_tokens=True))
except Exception as ex:
    print("generation test skipped:", str(ex)[:150])
print("SMOKE TEST OK" if SMOKE else "TRAINING DONE")


## 7. Evaluate the fine-tuned model
Same scorer and test set as the baselines.

In [ ]:
import os, sys, json, time, torch
from huggingface_hub import HfApi, login, snapshot_download
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

assert torch.cuda.is_available(), "GPU is off: Settings -> Accelerator -> GPU T4 x2"
DATA, MODEL = "/kaggle/working/data", "Qwen/Qwen2.5-3B-Instruct"
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi(); USER = api.whoami()["name"]
DATA_REPO, CKPT_REPO = f"{USER}/paperlens-data", f"{USER}/paperlens-ckpt"

if not os.path.exists(f"{DATA}/final/test.jsonl"):
    snapshot_download(DATA_REPO, repo_type="dataset", local_dir="/kaggle/working")
ADIR = "/kaggle/working/adapter_dl"
if not os.path.exists(f"{ADIR}/final/adapter_config.json"):
    snapshot_download(CKPT_REPO, repo_type="model", local_dir=ADIR, allow_patterns=["final/*"])
sys.path.insert(0, DATA)
import paperlens_eval as pe

# ---- load model exactly as in training: 4-bit base + the LoRA adapter ----
for n in ["model", "base_model", "opt"]: globals().pop(n, None)
import gc; gc.collect(); torch.cuda.empty_cache()
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
try: base_model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, dtype=torch.float16, device_map={"": 0})
except TypeError: base_model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, torch_dtype=torch.float16, device_map={"": 0})
model = PeftModel.from_pretrained(base_model, f"{ADIR}/final"); model.eval()

# ---- generate (resumable) ----
gold = [json.loads(l) for l in open(f"{DATA}/final/test.jsonl")]
out_path = f"{DATA}/preds_ft.jsonl"
def load_preds(p): return {json.loads(l)["id"]: json.loads(l)["pred"] for l in open(p)}
done = load_preds(out_path) if os.path.exists(out_path) else {}
todo = [r for r in gold if r["id"] not in done]
print(len(done), "already done,", len(todo), "to do")

BS, t0 = 8, time.time()
f = open(out_path, "a")
for i in range(0, len(todo), BS):
    batch = todo[i:i + BS]
    prompts = [tok.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True) for r in batch]
    enc = tok(prompts, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        out = model.generate(**enc, max_new_tokens=400, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    for r, t in zip(batch, texts):
        f.write(json.dumps({"id": r["id"], "pred": t}) + "\n"); done[r["id"]] = t
    f.flush(); print(f"{len(done)}/{len(gold)}  {time.time() - t0:.0f}s")
f.close()

# ---- comparison table ----
systems = {"base, short prompt": "preds_base.jsonl",
           "base, full-schema prompt": "preds_base_prompted.jsonl",
           "PaperLens (fine-tuned)": "preds_ft.jsonl"}
cols = ["valid", "task_f1", "method_f1", "datasets_f1_nonempty", "metrics_f1_nonempty", "type_acc", "result_num_f1"]
results = {}
for subset, label in [("all", "ALL 150 test rows"), ("reviewed", "REVIEWED 30 rows")]:
    print(f"\n=== {label} ===")
    print(f"{'system':28}" + "".join(f"{c[:14]:>16}" for c in cols))
    for name, fn in systems.items():
        p = f"{DATA}/{fn}"
        if not os.path.exists(p): print(f"{name:28} (missing {fn})"); continue
        res = pe.evaluate(load_preds(p), gold, subset); results[f"{name} | {subset}"] = res
        print(f"{name:28}" + "".join(f"{res[c]:>16.3f}" for c in cols))

json.dump(results, open(f"{DATA}/results.json", "w"), indent=1)
api.upload_folder(folder_path=DATA, repo_id=DATA_REPO, repo_type="dataset", path_in_repo="data")
print("\nbacked up")
bad = [r["id"] for r in gold if pe.parse(done[r["id"]]) is None]
print("invalid outputs:", len(bad), bad[:5])


## 8. Error analysis
Worst rows, contribution-type confusion, and a census of dataset, metric and main-result errors.

In [ ]:
import os, sys, json, re, collections
import numpy as np
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

W = "/kaggle/working"
need = [f"{W}/data/final/test.jsonl", f"{W}/data/preds_ft.jsonl", f"{W}/data/paperlens_eval.py"]
if not all(os.path.exists(p) for p in need):
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    snapshot_download("punit-09/paperlens-data", repo_type="dataset", local_dir=W,
                      allow_patterns=["data/final/*", "data/preds_*.jsonl",
                                      "data/results.json", "data/paperlens_eval.py"])
sys.path.insert(0, f"{W}/data")
import importlib, paperlens_eval as E
importlib.reload(E)

def rd(p): return [json.loads(l) for l in open(p) if l.strip()]
test  = rd(f"{W}/data/final/test.jsonl")
preds = {r["id"]: r["pred"] for r in rd(f"{W}/data/preds_ft.jsonl")}
print("test rows:", len(test), "| preds:", len(preds))

FIELDS = ["task_f1", "method_f1", "datasets_f1", "metrics_f1", "type_acc", "result_tok_f1"]
rows = []
for r in test:
    s = E.evaluate({r["id"]: preds[r["id"]]}, [r], subset="all")
    sc = {k: float(s[k]) for k in FIELDS}
    rows.append((float(np.mean(list(sc.values()))), sc, r))
rows.sort(key=lambda x: x[0])
print("macro (should be ~0.868):", round(np.mean([x[0] for x in rows]), 3))

# ---- worst rows (change START to see the next batch) ----
START, N = 5, 5
for rank, (m, sc, r) in enumerate(rows[START:START+N], START+1):
    user = r["messages"][1]["content"]
    gold = r["messages"][2]["content"]
    print("\n" + "="*100)
    print(f"#{rank} id={r['id']} macro={m:.3f} scores=" + json.dumps({k: round(v, 2) for k, v in sc.items()}))
    print(user)
    print("--- REFERENCE:", gold)
    print("--- PREDICTION:", preds[r["id"]])

# ---- contribution_type per-class accuracy + confusion ----
cm = collections.Counter(); sup = collections.Counter()
for r in test:
    g = json.loads(r["messages"][2]["content"])["contribution_type"]
    p = E.parse(preds[r["id"]]); p = p["contribution_type"] if p else "INVALID"
    cm[(g, p)] += 1; sup[g] += 1
print("\n" + "="*100 + "\ncontribution_type (gold -> pred):")
for g in sorted(sup):
    ok = cm[(g, g)]
    wrong = {p: c for (gg, p), c in cm.items() if gg == g and p != g}
    print(f"  {g:22s} n={sup[g]:3d} correct={ok:3d} acc={ok/sup[g]:.2f} wrong->{wrong}")


In [ ]:
import os, sys, json, re, collections
import numpy as np
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

W = "/kaggle/working"
need = [f"{W}/data/final/test.jsonl", f"{W}/data/preds_ft.jsonl", f"{W}/data/paperlens_eval.py"]
if not all(os.path.exists(p) for p in need):
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    snapshot_download("punit-09/paperlens-data", repo_type="dataset", local_dir=W,
                      allow_patterns=["data/final/*", "data/preds_*.jsonl", "data/paperlens_eval.py"])
sys.path.insert(0, f"{W}/data")
import paperlens_eval as E

def rd(p): return [json.loads(l) for l in open(p) if l.strip()]
test = rd(f"{W}/data/final/test.jsonl")
preds = {r["id"]: r["pred"] for r in rd(f"{W}/data/preds_ft.jsonl")}
norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())

recs = []
for r in test:
    g = json.loads(r["messages"][2]["content"])
    p = E.parse(preds[r["id"]])
    if p is None: continue
    title = r["messages"][1]["content"].split("\n")[0]
    recs.append((r["id"], title, g, p))
print("rows analysed:", len(recs))

# ---- datasets / metrics: false positives vs misses ----
for field in ["datasets", "metrics"]:
    print("\n" + "="*90 + f"\n{field.upper()}")
    exact = fp_rows = fn_rows = 0
    fp_tags = collections.Counter()
    lines = []
    for id_, title, g, p in recs:
        G = {norm(x): x for x in g[field]}
        P = {norm(x): x for x in p[field]}
        fp = [P[k] for k in P if k not in G]
        fn = [G[k] for k in G if k not in P]
        exact += (not fp and not fn); fp_rows += bool(fp); fn_rows += bool(fn)
        for x in fp:
            if norm(x) in norm(title): fp_tags["pred item appears in title"] += 1
            if norm(x) in norm(p["method"]) or norm(p["method"]) in norm(x): fp_tags["pred item ~ pred method"] += 1
        if fp or fn: lines.append(f"{id_} | FP={fp} | MISS={fn}")
    print(f"exact set match: {exact}/{len(recs)} | rows with extra items: {fp_rows} | rows with missed items: {fn_rows}")
    print("FP tags:", dict(fp_tags))
    print("\n".join(lines))

# ---- method ----
ex = cont = 0; lg = []; lp = []
for id_, title, g, p in recs:
    a, b = norm(g["method"]), norm(p["method"])
    ex += a == b; cont += (a in b or b in a)
    lg.append(len(g["method"].split())); lp.append(len(p["method"].split()))
print("\n" + "="*90 + f"\nMETHOD: exact {ex}/{len(recs)} | one contains the other {cont}/{len(recs)} | mean words gold {np.mean(lg):.1f} pred {np.mean(lp):.1f}")

# ---- main_result ----
dg = lambda s: bool(re.search(r"\d", s))
same = 0; g_num_p_no = 0; g_no_p_num = 0; wg = []; wp = []
for id_, title, g, p in recs:
    same += norm(g["main_result"]) == norm(p["main_result"])
    g_num_p_no += dg(g["main_result"]) and not dg(p["main_result"])
    g_no_p_num += (not dg(g["main_result"])) and dg(p["main_result"])
    wg.append(len(g["main_result"].split())); wp.append(len(p["main_result"].split()))
print(f"MAIN_RESULT: identical {same}/{len(recs)} | gold has digits but pred none: {g_num_p_no} | pred has digits but gold none: {g_no_p_num} | mean words gold {np.mean(wg):.1f} pred {np.mean(wp):.1f}")

# ---- how references treat papers' own benchmarks ----
print("\n" + "="*90 + "\nbenchmark_or_dataset rows (gold datasets vs pred datasets):")
for id_, title, g, p in recs:
    if g["contribution_type"] == "benchmark_or_dataset":
        print(f"{id_} | {title[:70]} | gold={g['datasets']} | pred={p['datasets']}")


## 9. Comparison with gpt-oss-20b (Groq) and latency
gpt-oss-20b with a plain prompt, then with the labeling rules, on the same 150 test rows; then fine-tuned model latency on a T4.

In [ ]:
import os, sys, json, time, re, requests, importlib
import numpy as np
from huggingface_hub import snapshot_download, HfApi
from kaggle_secrets import UserSecretsClient

W = "/kaggle/working"; D = f"{W}/data"
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
OUT = f"{D}/preds_gptoss.jsonl"
need = [f"{D}/final/test.jsonl", f"{D}/paperlens_eval.py", f"{D}/preds_ft.jsonl", f"{D}/preds_base_prompted.jsonl"]
if not all(os.path.exists(p) for p in need):
    snapshot_download("punit-09/paperlens-data", repo_type="dataset", local_dir=W,
                      allow_patterns=["data/final/*", "data/preds_*.jsonl", "data/paperlens_eval.py"])
sys.path.insert(0, D)
import paperlens_eval as E; importlib.reload(E)
def rd(p): return [json.loads(l) for l in open(p) if l.strip()]
test = rd(f"{D}/final/test.jsonl")

SYS = """You extract structured information from an ML paper (title + abstract).
Reply with ONE JSON object and nothing else (no markdown fences, no comments). Keys:
- task: string, lowercase noun phrase, max 10 words
- method: string, the authors' own proposed method (use its name if given), max 12 words
- datasets: list of strings, names of datasets or benchmarks mentioned; [] if none
- metrics: list of strings, named evaluation metrics mentioned; [] if none
- main_result: ONE sentence (a string) with the main empirical finding, keeping numbers
- contribution_type: exactly one of "new_method", "benchmark_or_dataset", "analysis", "survey", "other"
Use only information in the text."""

SECRET_NAME = "GROQ_API_KEY"
KEY = UserSecretsClient().get_secret(SECRET_NAME)
URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL = "openai/gpt-oss-20b"
sess = requests.Session()

def secs(s):
    s = (s or "").strip()
    try:
        if s.endswith("ms"): return float(s[:-2]) / 1000
        m = re.fullmatch(r"(?:(\d+)m)?(?:([\d.]+)s)?", s)
        return int(m.group(1) or 0) * 60 + float(m.group(2) or 0)
    except Exception:
        return 10.0

def call(user):
    body = {"model": MODEL, "temperature": 0, "max_completion_tokens": 2048, "reasoning_effort": "low",
            "messages": [{"role": "system", "content": SYS}, {"role": "user", "content": user}]}
    for attempt in range(1, 7):
        t0 = time.time()
        try:
            resp = sess.post(URL, headers={"Authorization": f"Bearer {KEY}"}, json=body, timeout=90)
        except requests.RequestException:
            time.sleep(min(60, 5 * attempt)); continue
        dt = time.time() - t0; h = resp.headers
        if resp.status_code == 200:
            j = resp.json(); ch = j["choices"][0]; u = j.get("usage", {})
            rec = dict(pred=ch["message"].get("content") or "", latency_s=round(dt, 3), attempts=attempt,
                       finish=ch.get("finish_reason"), prompt_tokens=u.get("prompt_tokens"),
                       completion_tokens=u.get("completion_tokens"),
                       reasoning_tokens=(u.get("completion_tokens_details") or {}).get("reasoning_tokens"))
            rem = int(h.get("x-ratelimit-remaining-tokens", 8000)); lim = int(h.get("x-ratelimit-limit-tokens", 8000))
            if rem < 1500:                       # wait until ~1500 tokens are available again
                time.sleep((1500 - rem) / (lim / 60) + 1)
            return rec
        if resp.status_code == 429:
            time.sleep(max(secs(h.get("retry-after", "20") + "s"), 5) + 1); continue
        if resp.status_code >= 500:
            time.sleep(min(60, 5 * attempt)); continue
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:300]}")   # 400/401/404: stop, don't retry
    return None

done = {}
if os.path.exists(OUT):
    for r in rd(OUT): done[r["id"]] = r
todo = [r for r in test if r["id"] not in done]
print(f"already done: {len(done)} | to do: {len(todo)}")

api = HfApi(token=HF_TOKEN)
t_start = time.time()
try:
    with open(OUT, "a") as f:
        for i, r in enumerate(todo, 1):
            rec = call(r["messages"][1]["content"])
            if rec is None:
                print("gave up on", r["id"], "(will be retried on rerun)"); continue
            rec["id"] = r["id"]; done[r["id"]] = rec
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
            if i % 10 == 0: print(f"{i}/{len(todo)} | {time.time()-t_start:.0f}s elapsed")
finally:
    if os.path.exists(OUT):
        api.upload_file(path_or_fileobj=OUT, path_in_repo="data/preds_gptoss.jsonl",
                        repo_id="punit-09/paperlens-data", repo_type="dataset",
                        commit_message="gpt-oss-20b (Groq) predictions")
        print("uploaded preds_gptoss.jsonl with", len(done), "rows")

# ------------------------- evaluation -------------------------
if len(done) < len(test):
    print(f"\nonly {len(done)}/{len(test)} rows done; rerun this cell to finish, evaluation runs after.")
else:
    norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
    systems = {
        "baseline (Qwen prompted)": {r["id"]: r["pred"] for r in rd(f"{D}/preds_base_prompted.jsonl")},
        "gpt-oss-20b (Groq, prompted)": {i: d["pred"] for i, d in done.items()},
        "PaperLens (fine-tuned)": {r["id"]: r["pred"] for r in rd(f"{D}/preds_ft.jsonl")},
    }
    FIELDS = ["task_f1", "method_f1", "datasets_f1", "metrics_f1", "type_acc", "result_tok_f1"]
    rows_by = {}
    for name, P in systems.items():
        rows_by[name] = np.array([[float(E.evaluate({r["id"]: P[r["id"]]}, [r], "all")[k]) for k in FIELDS] for r in test])
    rng = np.random.default_rng(0); idx = rng.integers(0, len(test), (2000, len(test)))
    def ci(v): b = v[idx].mean(1); return np.percentile(b, [2.5, 97.5])
    macro = {n: a.mean(1) for n, a in rows_by.items()}

    print("\n" + "=" * 100)
    print(f"{'system':32s} valid  macro [95% CI]          " + " ".join(f"{k[:8]:>8s}" for k in FIELDS))
    for n, a in rows_by.items():
        full = E.evaluate(systems[n], test, "all")
        lo, hi = ci(macro[n])
        print(f"{n:32s} {float(full['valid']):.3f}  {macro[n].mean():.3f} [{lo:.3f}, {hi:.3f}]  " + " ".join(f"{v:8.3f}" for v in a.mean(0)))
    print("\nnon-empty F1 / numbers:")
    for n in systems:
        full = E.evaluate(systems[n], test, "all")
        print(f"  {n:32s}", {k: round(float(full[k]), 3) for k in ["datasets_f1_nonempty", "metrics_f1_nonempty", "result_num_f1"]})
    print("\npaired differences in macro (95% CI):")
    ours, gpt, base = "PaperLens (fine-tuned)", "gpt-oss-20b (Groq, prompted)", "baseline (Qwen prompted)"
    for a_, b_ in [(ours, gpt), (gpt, base), (ours, base)]:
        d = macro[a_] - macro[b_]; lo, hi = ci(d)
        print(f"  {a_} - {b_}: {d.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]")

    print("\nhallucination (grounded = appears in title+abstract after lowercasing/alphanumeric normalisation):")
    for n, P in systems.items():
        tot = bad = ntot = nbad = 0
        for r in test:
            p = E.parse(P[r["id"]])
            if not p: continue
            txt = r["messages"][1]["content"]; tn = norm(txt)
            for it in p["datasets"] + p["metrics"]:
                tot += 1; bad += norm(it) not in tn
            for num in re.findall(r"\d+(?:\.\d+)?", p["main_result"]):
                ntot += 1; nbad += num not in txt
        print(f"  {n:32s} datasets/metrics not in text: {bad}/{tot} | numbers not in text: {nbad}/{ntot}")

    L = np.array([d["latency_s"] for d in done.values()]); pt = [d["prompt_tokens"] for d in done.values()]
    ct = [d["completion_tokens"] for d in done.values()]; rt = [d["reasoning_tokens"] or 0 for d in done.values()]
    print(f"\ngpt-oss-20b via Groq: latency median {np.median(L):.2f}s mean {L.mean():.2f}s max {L.max():.1f}s | "
          f"tokens total prompt {sum(pt)} completion {sum(ct)} (reasoning {sum(rt)}) | "
          f"finish reasons {dict(__import__('collections').Counter(d['finish'] for d in done.values()))} | "
          f"retried rows {sum(d['attempts']>1 for d in done.values())}")


In [ ]:
import os, sys, json, time, re, requests, importlib, collections
import numpy as np
from huggingface_hub import snapshot_download, HfApi
from kaggle_secrets import UserSecretsClient

W = "/kaggle/working"; D = f"{W}/data"
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN"); os.environ["HF_TOKEN"] = HF_TOKEN
OUT = f"{D}/preds_gptoss_rules.jsonl"
need = [f"{D}/final/test.jsonl", f"{D}/paperlens_eval.py", f"{D}/preds_ft.jsonl",
        f"{D}/preds_base_prompted.jsonl", f"{D}/preds_gptoss.jsonl"]
if not all(os.path.exists(p) for p in need):
    snapshot_download("punit-09/paperlens-data", repo_type="dataset", local_dir=W,
                      allow_patterns=["data/final/*", "data/preds_*.jsonl", "data/paperlens_eval.py"])
sys.path.insert(0, D)
import paperlens_eval as E; importlib.reload(E)
def rd(p): return [json.loads(l) for l in open(p) if l.strip()]
test = rd(f"{D}/final/test.jsonl")

SYS = """You extract structured information from an ML paper (title + abstract).
Reply with ONE JSON object and nothing else (no markdown fences, no comments), with exactly these keys in this order:
1. task: a lowercase noun phrase, max 10 words.
2. method: the authors' own proposed approach, max 12 words. If both a full name and an acronym are given, write "Full Name (ACRONYM)". Do not name baselines or components borrowed from others.
3. datasets: list of named datasets or benchmarks that appear in the title or abstract. Never model names (e.g. ResNet, BERT, Llama, GPT, ViT, Qwen). Do not include counts or descriptions such as "20 tissue sample pairs". Semantic-textual-similarity task names are not datasets. Use [] if none.
4. metrics: list of named evaluation metrics that appear in the title or abstract, at most 5. Not vague concepts such as cost, FPS, latency, throughput, speed, or efficiency. Use [] if none.
5. main_result: ONE sentence (a string). Copy numbers exactly as written in the abstract; never introduce a number that is not in the abstract.
6. contribution_type: exactly one of "new_method", "benchmark_or_dataset", "analysis", "survey", "other".
Use only information in the text. Never invent dataset or metric names."""

KEY = UserSecretsClient().get_secret("GROQ_API_KEY")   # edit if your secret has another name
URL = "https://api.groq.com/openai/v1/chat/completions"; MODEL = "openai/gpt-oss-20b"
sess = requests.Session(); shown429 = [0]

def secs(s):
    s = (s or "").strip()
    try:
        if s.endswith("ms"): return float(s[:-2]) / 1000
        m = re.fullmatch(r"(?:(\d+)m)?(?:([\d.]+)s)?", s)
        return int(m.group(1) or 0) * 60 + float(m.group(2) or 0)
    except Exception: return 10.0

def call(user):
    body = {"model": MODEL, "temperature": 0, "max_completion_tokens": 2048, "reasoning_effort": "low",
            "messages": [{"role": "system", "content": SYS}, {"role": "user", "content": user}]}
    for attempt in range(1, 7):
        t0 = time.time()
        try: resp = sess.post(URL, headers={"Authorization": f"Bearer {KEY}"}, json=body, timeout=90)
        except requests.RequestException: time.sleep(min(60, 5 * attempt)); continue
        dt = time.time() - t0; h = resp.headers
        if resp.status_code == 200:
            j = resp.json(); ch = j["choices"][0]; u = j.get("usage", {})
            rec = dict(pred=ch["message"].get("content") or "", latency_s=round(dt, 3), attempts=attempt,
                       finish=ch.get("finish_reason"), prompt_tokens=u.get("prompt_tokens"),
                       completion_tokens=u.get("completion_tokens"),
                       reasoning_tokens=(u.get("completion_tokens_details") or {}).get("reasoning_tokens"))
            rem = int(h.get("x-ratelimit-remaining-tokens", 8000)); lim = int(h.get("x-ratelimit-limit-tokens", 8000))
            if rem < 2500: time.sleep((2500 - rem) / (lim / 60) + 1)
            return rec
        if resp.status_code == 429:
            if shown429[0] < 3:
                shown429[0] += 1
                print("429:", resp.text[:250], "| retry-after:", h.get("retry-after"),
                      "| remaining tokens header:", h.get("x-ratelimit-remaining-tokens"))
            time.sleep(max(secs(str(h.get("retry-after", "20")) + "s"), 5) + 1); continue
        if resp.status_code >= 500: time.sleep(min(60, 5 * attempt)); continue
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:300]}")
    return None

done = {r["id"]: r for r in rd(OUT)} if os.path.exists(OUT) else {}
todo = [r for r in test if r["id"] not in done]
print(f"already done: {len(done)} | to do: {len(todo)}")
api = HfApi(token=HF_TOKEN); t_start = time.time()
try:
    with open(OUT, "a") as f:
        for i, r in enumerate(todo, 1):
            rec = call(r["messages"][1]["content"])
            if rec is None: print("gave up on", r["id"], "(retried on rerun)"); continue
            rec["id"] = r["id"]; done[r["id"]] = rec
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
            if i % 10 == 0: print(f"{i}/{len(todo)} | {time.time()-t_start:.0f}s elapsed")
finally:
    if os.path.exists(OUT):
        api.upload_file(path_or_fileobj=OUT, path_in_repo="data/preds_gptoss_rules.jsonl",
                        repo_id="punit-09/paperlens-data", repo_type="dataset",
                        commit_message="gpt-oss-20b (Groq) with rules prompt")
        print("uploaded preds_gptoss_rules.jsonl with", len(done), "rows")

if len(done) < len(test):
    print(f"\nonly {len(done)}/{len(test)} rows done; rerun this cell to finish, evaluation runs after.")
else:
    norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
    P = lambda fn: {r["id"]: r["pred"] for r in rd(f"{D}/{fn}")}
    systems = {"baseline (Qwen prompted)": P("preds_base_prompted.jsonl"),
               "gpt-oss-20b plain prompt": P("preds_gptoss.jsonl"),
               "gpt-oss-20b rules prompt": {i: d["pred"] for i, d in done.items()},
               "PaperLens (fine-tuned)": P("preds_ft.jsonl")}
    FIELDS = ["task_f1", "method_f1", "datasets_f1", "metrics_f1", "type_acc", "result_tok_f1"]
    rows_by = {n: np.array([[float(E.evaluate({r["id"]: p[r["id"]]}, [r], "all")[k]) for k in FIELDS] for r in test])
               for n, p in systems.items()}
    idx = np.random.default_rng(0).integers(0, len(test), (2000, len(test)))
    ci = lambda v: np.percentile(v[idx].mean(1), [2.5, 97.5])
    macro = {n: a.mean(1) for n, a in rows_by.items()}
    print("\n" + "=" * 100)
    print(f"{'system':28s} valid  macro [95% CI]          " + " ".join(f"{k[:8]:>8s}" for k in FIELDS))
    for n, a in rows_by.items():
        full = E.evaluate(systems[n], test, "all"); lo, hi = ci(macro[n])
        print(f"{n:28s} {float(full['valid']):.3f}  {macro[n].mean():.3f} [{lo:.3f}, {hi:.3f}]  " + " ".join(f"{v:8.3f}" for v in a.mean(0)))
    print("\nnon-empty F1 / numbers:")
    for n in systems:
        full = E.evaluate(systems[n], test, "all")
        print(f"  {n:28s}", {k: round(float(full[k]), 3) for k in ["datasets_f1_nonempty", "metrics_f1_nonempty", "result_num_f1"]})
    print("\npaired macro differences (95% CI):")
    for a_, b_ in [("PaperLens (fine-tuned)", "gpt-oss-20b rules prompt"),
                   ("gpt-oss-20b rules prompt", "gpt-oss-20b plain prompt")]:
        d = macro[a_] - macro[b_]; lo, hi = ci(d); print(f"  {a_} - {b_}: {d.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]")
    print("\nper-field paired difference PaperLens - gpt-oss rules (95% CI):")
    for j, k in enumerate(FIELDS):
        d = rows_by["PaperLens (fine-tuned)"][:, j] - rows_by["gpt-oss-20b rules prompt"][:, j]; lo, hi = ci(d)
        print(f"  {k:14s} {d.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]")
    print("\nnot found verbatim in text:")
    for n, Pn in systems.items():
        tot = bad = ntot = nbad = 0
        for r in test:
            p = E.parse(Pn[r["id"]])
            if not p: continue
            txt = r["messages"][1]["content"]; tn = norm(txt)
            for it in p["datasets"] + p["metrics"]: tot += 1; bad += norm(it) not in tn
            for num in re.findall(r"\d+(?:\.\d+)?", p["main_result"]): ntot += 1; nbad += num not in txt
        print(f"  {n:28s} datasets/metrics {bad}/{tot} | numbers {nbad}/{ntot}")
    dd = list(done.values())
    print(f"\nrules run: prompt tokens {sum(d['prompt_tokens'] for d in dd)}, completion {sum(d['completion_tokens'] for d in dd)}, "
          f"finish {dict(collections.Counter(d['finish'] for d in dd))}, retried rows {sum(d['attempts']>1 for d in dd)}")


In [ ]:
import os, sys, json, time, gc
import numpy as np
import torch
from huggingface_hub import HfApi, login, snapshot_download
from kaggle_secrets import UserSecretsClient
import transformers, peft, bitsandbytes
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

assert torch.cuda.is_available(), "GPU is off"
print("versions:", transformers.__version__, peft.__version__, bitsandbytes.__version__, torch.__version__)
gpu = torch.cuda.get_device_name(0); print("GPU:", gpu)

W = "/kaggle/working"; DATA = f"{W}/data"; MODEL = "Qwen/Qwen2.5-3B-Instruct"
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN"); login(HF_TOKEN)
api = HfApi(); USER = api.whoami()["name"]
DATA_REPO, CKPT_REPO = f"{USER}/paperlens-data", f"{USER}/paperlens-ckpt"
if not all(os.path.exists(f"{DATA}/{p}") for p in ["final/test.jsonl", "preds_ft.jsonl", "paperlens_eval.py"]):
    snapshot_download(DATA_REPO, repo_type="dataset", local_dir=W,
                      allow_patterns=["data/final/*", "data/preds_*.jsonl", "data/paperlens_eval.py"])
ADIR = f"{W}/adapter_dl"
if not os.path.exists(f"{ADIR}/final/adapter_config.json"):
    snapshot_download(CKPT_REPO, repo_type="model", local_dir=ADIR, allow_patterns=["final/*"])
sys.path.insert(0, DATA)
import paperlens_eval as pe

# ---- load exactly as in your inference cell ----
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
try: base_model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, dtype=torch.float16, device_map={"": 0})
except TypeError: base_model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, torch_dtype=torch.float16, device_map={"": 0})
model = PeftModel.from_pretrained(base_model, f"{ADIR}/final"); model.eval()
mem_load = torch.cuda.max_memory_allocated() / 1e9
print(f"peak GPU memory after loading: {mem_load:.2f} GB")

gold = [json.loads(l) for l in open(f"{DATA}/final/test.jsonl") if l.strip()]
saved = {json.loads(l)["id"]: json.loads(l)["pred"] for l in open(f"{DATA}/preds_ft.jsonl") if l.strip()}

def gen(rows):
    prompts = [tok.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True) for r in rows]
    enc = tok(prompts, return_tensors="pt", padding=True).to("cuda")
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        out = model.generate(**enc, max_new_tokens=400, do_sample=False, pad_token_id=tok.pad_token_id, use_cache=True)
    torch.cuda.synchronize(); dt = time.perf_counter() - t0
    plen = enc["input_ids"].shape[1]
    texts = tok.batch_decode(out[:, plen:], skip_special_tokens=True)
    return texts, dt, plen, out.shape[1] - plen

N = 30
rows = gold[:N]
for r in rows[:2]: gen([r])                      # warm-up (not timed)

# ---- batch size 1: per-request latency ----
torch.cuda.reset_peak_memory_stats()
lat, ntok, same, valid = [], [], 0, 0
for i, r in enumerate(rows, 1):
    t, dt, plen, nnew = gen([r])
    lat.append(dt); ntok.append(nnew)
    same += t[0].strip() == saved[r["id"]].strip()
    valid += pe.parse(t[0]) is not None
    if i % 10 == 0: print(f"{i}/{N} done")
lat, ntok = np.array(lat), np.array(ntok)
mem_gen1 = torch.cuda.max_memory_allocated() / 1e9

# ---- batch size 8: throughput (16 rows) ----
torch.cuda.reset_peak_memory_stats(); tb = 0.0
for i in range(0, 16, 8):
    _, dt, _, _ = gen(rows[i:i+8]); tb += dt
mem_gen8 = torch.cuda.max_memory_allocated() / 1e9

res = {"gpu": gpu, "n_rows": N, "versions": {"transformers": transformers.__version__, "peft": peft.__version__,
        "bitsandbytes": bitsandbytes.__version__, "torch": torch.__version__},
       "bs1_latency_s": {"median": float(np.median(lat)), "mean": float(lat.mean()),
                         "p90": float(np.percentile(lat, 90)), "max": float(lat.max())},
       "mean_new_tokens": float(ntok.mean()), "median_tokens_per_s_bs1": float(np.median(ntok / lat)),
       "bs8_rows_per_s": float(16 / tb), "peak_mem_gb": {"load": mem_load, "bs1": mem_gen1, "bs8": mem_gen8},
       "identical_to_saved_preds": f"{same}/{N}", "valid_json": f"{valid}/{N}"}
print(json.dumps(res, indent=1))
json.dump(res, open(f"{DATA}/latency_ft.json", "w"), indent=1)
api.upload_file(path_or_fileobj=f"{DATA}/latency_ft.json", path_in_repo="data/latency_ft.json",
                repo_id=DATA_REPO, repo_type="dataset", commit_message="fine-tuned latency on T4")
print("uploaded latency_ft.json")


## 10. Final results
Per-system scores, 95% bootstrap confidence intervals, paired differences and the grounding check. Writes `results_final.json` and `results_table.md`.

In [ ]:
import os, sys, json, re
import numpy as np
from huggingface_hub import HfApi, login, snapshot_download
from kaggle_secrets import UserSecretsClient

DATA = "/kaggle/working/data"
login(UserSecretsClient().get_secret("HF_TOKEN"))
api = HfApi(); REPO = f"{api.whoami()['name']}/paperlens-data"
need = ["preds_base_prompted.jsonl", "preds_ft.jsonl", "preds_gptoss.jsonl",
        "preds_gptoss_rules.jsonl", "paperlens_eval.py", "final/test.jsonl"]
if not all(os.path.exists(f"{DATA}/{f}") for f in need):
    snapshot_download(REPO, repo_type="dataset", local_dir="/kaggle/working")
missing = [f for f in need if not os.path.exists(f"{DATA}/{f}")]
assert not missing, f"missing on HF: {missing}"
sys.path.insert(0, DATA)
import paperlens_eval as pe

gold = [json.loads(l) for l in open(f"{DATA}/final/test.jsonl")]
G = {r["id"]: json.loads(r["messages"][-1]["content"]) for r in gold}
SRC = {r["id"]: r["messages"][1]["content"] for r in gold}
REV = np.array([bool(r.get("reviewed")) for r in gold])
n = len(gold)

def load_preds(fn):
    out = {}
    for l in open(f"{DATA}/{fn}"):
        d = json.loads(l)
        t = d.get("pred", d.get("output", d.get("text", "")))
        out[d["id"]] = t if isinstance(t, str) else json.dumps(t)
    return out

FIELDS = ["task", "method", "datasets", "metrics", "type", "result"]
def row_scores(text, g):
    p = pe.parse(text)
    if p is None: return [0.0] * 6, None
    return [pe.tok_f1(p["task"], g["task"]), pe.tok_f1(p["method"], g["method"]),
            pe.set_f1(p["datasets"], g["datasets"]), pe.set_f1(p["metrics"], g["metrics"]),
            float(p["contribution_type"] == g["contribution_type"]),
            pe.tok_f1(p["main_result"], g["main_result"])], p

def grounded(item, text):
    c = pe.compact(item)
    if not c: return True
    if len(c) >= 4: return c in pe.compact(text)
    return re.search(r"(?<![A-Za-z0-9])" + re.escape(item.strip()) + r"(?![A-Za-z0-9])", text) is not None

systems = {"Qwen2.5-3B prompted (baseline)": "preds_base_prompted.jsonl",
           "gpt-oss-20b, plain prompt": "preds_gptoss.jsonl",
           "gpt-oss-20b, rules prompt": "preds_gptoss_rules.jsonl",
           "PaperLens (fine-tuned)": "preds_ft.jsonl"}
S = {}
for name, fn in systems.items():
    P = load_preds(fn)
    print(f"{name}: {sum(bool(P.get(r['id'])) for r in gold)}/{n} rows have output")
    sc, valid, hi, hn = [], [], [0, 0], [0, 0]
    for r in gold:
        s, p = row_scores(P.get(r["id"], ""), G[r["id"]]); sc.append(s); valid.append(p is not None)
        if p is not None:
            items = p["datasets"] + p["metrics"]
            hi[0] += sum(not grounded(x, SRC[r["id"]]) for x in items); hi[1] += len(items)
            ns = pe.nums(p["main_result"])
            hn[0] += sum(x not in SRC[r["id"]] for x in ns); hn[1] += len(ns)
    S[name] = dict(sc=np.array(sc), valid=np.array(valid), hi=hi, hn=hn)

idx = np.random.default_rng(0).integers(0, n, size=(2000, n))
def ci(v):
    m = v[idx].mean(axis=1)
    return float(v.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

res, md = {"n_test": n, "n_reviewed": int(REV.sum()), "systems": {}, "paired_vs_paperlens": {}}, [
    "| system | valid JSON | macro (95% CI) | task | method | datasets | metrics | type | result | datasets/metrics not found in abstract |",
    "|---|---|---|---|---|---|---|---|---|---|"]
for name, d in S.items():
    m, lo, hi_ = ci(d["sc"].mean(axis=1)); f = d["sc"].mean(axis=0)
    res["systems"][name] = dict(valid=float(d["valid"].mean()), macro=[m, lo, hi_],
        fields={k: float(f[i]) for i, k in enumerate(FIELDS)},
        reviewed_slice_macro=float(d["sc"].mean(axis=1)[REV].mean()),
        ungrounded_datasets_metrics=d["hi"], ungrounded_numbers=d["hn"])
    md.append(f"| {name} | {d['valid'].mean():.3f} | {m:.3f} [{lo:.3f}, {hi_:.3f}] | "
              + " | ".join(f"{x:.3f}" for x in f) + f" | {d['hi'][0]}/{d['hi'][1]} |")

ftn = "PaperLens (fine-tuned)"; ft = S[ftn]["sc"]
for name, d in S.items():
    if name == ftn: continue
    out = {"macro": ci(ft.mean(axis=1) - d["sc"].mean(axis=1))}
    for i, k in enumerate(FIELDS): out[k] = ci(ft[:, i] - d["sc"][:, i])
    res["paired_vs_paperlens"][name] = out

lat = f"{DATA}/latency_ft.json"
if os.path.exists(lat): res["latency_ft_t4"] = json.load(open(lat))

print("\n" + "\n".join(md))
print("\nPaperLens minus each system (95% bootstrap CI; ties = CI includes 0):")
for name, out in res["paired_vs_paperlens"].items():
    print(f"\nvs {name}")
    for k, (m, lo, hi_) in out.items(): print(f"  {k:9} {m:+.3f} [{lo:+.3f}, {hi_:+.3f}]")
print(f"\nReviewed-slice macro (n={int(REV.sum())}, noisy):",
      {k: round(v['reviewed_slice_macro'], 3) for k, v in res['systems'].items()})

json.dump(res, open(f"{DATA}/results_final.json", "w"), indent=1)
open(f"{DATA}/results_table.md", "w").write("\n".join(md) + "\n")
api.upload_folder(folder_path=DATA, repo_id=REPO, repo_type="dataset", path_in_repo="data")
print("\nsaved results_final.json + results_table.md and backed up")
